# 03 — LoRA Training (Google Colab)

This notebook trains a **Ravi Varma-inspired LoRA** on top of Stable Diffusion 1.5 using the project training pipeline.

**Recommended runtime:** Google Colab GPU — NVIDIA T4 or better.

### Project status
- Dataset preparation: completed
- LoRA training: completed
- Preliminary generation: completed
- ControlNet / evaluation / dashboard: future work

> Generated images are AI-generated Ravi Varma-inspired artwork, not authentic paintings by Raja Ravi Varma.

## 1. Check the GPU


In [ ]:
!nvidia-smi

## 2. Get the project and install dependencies

If the project has already been uploaded to Colab, skip the clone command and only run the installation commands.


In [ ]:
# If starting from GitHub in a fresh Colab runtime, uncomment the next two lines.
# !git clone https://github.com/f20240684-cmd/the-next-ravi-varma.git
# %cd the-next-ravi-varma

%cd /content/the-next-ravi-varma
!pip install -q -r requirements-colab.txt
!pip install -q -e .

## 3. Colab compatibility fix

The Colab environment used for this project initially had an incompatible `torchao` version. The following pins the compatible version without replacing the Colab PyTorch installation.


In [ ]:
!pip install -q --upgrade --no-deps "torchao==0.18.0"

import torch
import torchao
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("torchao:", torchao.__version__)

## 4. Mount Google Drive (optional)

Use this only if the dataset or checkpoints are stored in Google Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Example:
# !cp -r /content/drive/MyDrive/ravi_varma_data/* data/

## 5. Validate → caption → prepare the dataset

The final training dataset used for the project contains **39 valid images** after validation.

In [ ]:
!python scripts/validate_dataset.py
!python scripts/generate_captions.py --write-metadata
!python scripts/prepare_dataset.py

## 6. Run a short LoRA smoke test

Run a short training job first to verify that the training stack, dataset, LoRA configuration and GPU are working correctly.

In [ ]:
!python scripts/train_lora.py --config configs/lora_sd15.yaml --max-train-steps 50

## 7. Full LoRA training

After the smoke test succeeds, run the complete training configuration.

The project configuration uses:
- Stable Diffusion 1.5
- LoRA rank: 16
- Trigger token: `<rvvarma>`
- Batch size: 1
- Gradient accumulation: 4
- FP16 mixed precision
- Gradient checkpointing
- Maximum training steps: 1500

In [ ]:
!python scripts/train_lora.py --config configs/lora_sd15.yaml

## 8. Generate a preliminary result

Use the trained LoRA to generate a new scene. The generation pipeline loads the trained checkpoint and records generation metadata.

In [ ]:
!python scripts/generate.py --prompt "<rvvarma>, Arjuna standing before Krishna on the battlefield of Kurukshetra, traditional Indian attire rendered with fine textile details, oil painting"

## 9. Try additional prompts

The following prompts can be used to demonstrate that the trained model can generate different scenes while using the learned visual style.

In [ ]:
prompts = [
    "<rvvarma>, royal Indian woman in traditional silk attire, ornate jewelry, palace interior, oil painting",
    "<rvvarma>, mythological Indian prince standing in a palace garden, traditional clothing, dramatic lighting, oil painting",
    "<rvvarma>, Indian goddess seated on a lotus, richly detailed traditional attire, classical oil painting",
]

for prompt in prompts:
    print("Generating:", prompt)
    !python scripts/generate.py --prompt "$prompt"

## 10. Download the trained LoRA checkpoint

The checkpoint is packaged so it can be moved back to the local project or stored separately.

In [ ]:
from google.colab import files
import shutil

shutil.make_archive(
    'ravi_varma_lora',
    'zip',
    'checkpoints/lora/ravi_varma'
)

files.download('ravi_varma_lora.zip')

## 11. Next steps

After training and preliminary generation:

1. Copy the trained checkpoint into `checkpoints/lora/ravi_varma/` on the local project if required.
2. Continue with prompt expansion improvements.
3. Add ControlNet conditioning.
4. Run image/style evaluation.
5. Integrate the final pipeline into the Gradio dashboard.

These stages are part of the remaining project work.